# Module 06 Lab — AutoGen: Multi-Agent GroupChat

**Course:** AI Agents Teaching Platform  
**Module:** 06 — Multi-Agent Systems  
**Estimated time:** ~90 minutes

---

### What you'll build

A research GroupChat with four specialist agents:
- **Orchestrator** — decomposes tasks and synthesises final answer
- **Literature Reviewer** — finds and summarises existing research
- **Critic** — challenges assumptions, finds gaps
- **Formatter** — structures the final report

The chat terminates when any agent says `FINAL_ANSWER:`.

> **Note:** This lab uses `langchain-anthropic` for LLM access via AutoGen's LiteLLM bridge. Set `ANTHROPIC_API_KEY` below.

In [ ]:
# Note: pyautogen v0.2.x is the stable API used in this lab
%pip install -q pyautogen

In [ ]:
import os
from getpass import getpass

os.environ["ANTHROPIC_API_KEY"] = getpass("Paste your Anthropic API key: ")
print("Key set ✓")

## Step 1 — Configure AutoGen's LLM

AutoGen uses a config list to route requests. We point it at the Anthropic API.

In [ ]:
import autogen

llm_config = {
    "config_list": [
        {
            "model": "claude-haiku-4-5-20251001",
            "api_key": os.environ["ANTHROPIC_API_KEY"],
            "api_type": "anthropic",
        }
    ],
    "temperature": 0.3,
    "max_tokens": 512,
}

print("LLM config set ✓")

## Step 2 — Define the agents (TODO)

Four `AssistantAgent` instances, each with a distinct `system_message` defining their role.
Plus a `UserProxyAgent` with `human_input_mode="NEVER"` (so the chat runs fully automated).

**TODO:** Fill in system messages for the Critic and Formatter agents.
Keep them under 100 words — tight, concrete, and behaviour-first.

In [ ]:
orchestrator = autogen.AssistantAgent(
    name="Orchestrator",
    llm_config=llm_config,
    system_message="""You coordinate the research team. When given a question:
1. First message: break it into 2–3 sub-questions for the Literature_Reviewer.
2. After literature review: ask Critic to challenge the findings.
3. After critique: ask Formatter to produce the final report.
4. Once formatted: say FINAL_ANSWER: followed by the full synthesised answer.
Keep your own messages brief — you direct, you don't research."""
)

literature_reviewer = autogen.AssistantAgent(
    name="Literature_Reviewer",
    llm_config=llm_config,
    system_message="""You find and summarise relevant research on the sub-questions you receive.
For each sub-question, provide 2–3 key findings with reasoning.
Be specific: cite concepts, mechanisms, or known results.
Label your output clearly with the sub-question before each section.
Do NOT produce a final answer — that is the Orchestrator's job."""
)

critic = autogen.AssistantAgent(
    name="Critic",
    llm_config=llm_config,
    system_message="""TODO: Write a system message for the Critic.
It should: challenge assumptions, identify gaps, point out what's missing or oversimplified.
Tone: constructive, precise. Length: under 80 words."""
)

formatter = autogen.AssistantAgent(
    name="Formatter",
    llm_config=llm_config,
    system_message="""TODO: Write a system message for the Formatter.
It should: take all prior findings and critique, structure them into a clear report
with headings, a brief summary, and a 'Limitations' section.
Length: under 60 words."""
)

user_proxy = autogen.UserProxyAgent(
    name="User",
    human_input_mode="NEVER",
    max_consecutive_auto_reply=12,
    is_termination_msg=lambda msg: "FINAL_ANSWER:" in msg.get("content", ""),
    code_execution_config=False,
)

print("Agents defined ✓")

<details>
<summary>💡 Stuck? Reveal system message solutions</summary>

```python
# Critic
system_message="""You are a rigorous critic. When you receive literature findings:
- Challenge each claim: what evidence is missing?
- Identify contradictions or oversimplifications.
- Suggest 1–2 gaps that future research should address.
Be constructive — point out what's weak and why. Do NOT rewrite the findings."""

# Formatter
system_message="""You produce the final structured report. Given findings and critique:
- Write a 2-paragraph Summary
- Key Findings (bullet points)
- Limitations (from the Critic)
Keep it under 350 words. Plain markdown, no inline code blocks."""
```

</details>

## Step 3 — Create the GroupChat (TODO)

Wire up a `GroupChat` with all five participants and a `GroupChatManager`.

**TODO:** Create the `GroupChat` with:
- `agents` in a sensible order
- `max_round=12` to cap the conversation
- `speaker_selection_method="auto"` (LLM picks who speaks next)

Then create a `GroupChatManager` wrapping it.

In [ ]:
# TODO: create groupchat and manager
# groupchat = autogen.GroupChat(agents=[...], max_round=12, speaker_selection_method="auto")
# manager = autogen.GroupChatManager(groupchat=groupchat, llm_config=llm_config)

groupchat = None  # replace me
manager = None    # replace me

print("GroupChat defined ✓" if manager else "TODO: complete the GroupChat setup")

<details>
<summary>💡 Stuck? Reveal the GroupChat solution</summary>

```python
groupchat = autogen.GroupChat(
    agents=[user_proxy, orchestrator, literature_reviewer, critic, formatter],
    max_round=12,
    speaker_selection_method="auto",
)
manager = autogen.GroupChatManager(groupchat=groupchat, llm_config=llm_config)
```

The order in `agents` is the default speaking order when `speaker_selection_method="round_robin"`.
With `"auto"`, the manager LLM decides who speaks based on context.

</details>

## Step 4 — Run the GroupChat

In [ ]:
QUESTION = "What are the key challenges in deploying large language models in production systems?"

# This initiates the multi-agent conversation
user_proxy.initiate_chat(
    manager,
    message=QUESTION,
)

print("\n--- Chat complete ---")

## Step 5 — Inspect the chat history

In [ ]:
history = user_proxy.chat_messages.get(manager, [])
print(f"Total messages: {len(history)}")
for msg in history:
    role = msg.get("name", msg.get("role", "?"))
    content = msg.get("content", "")[:120]
    print(f"\n[{role}]: {content}..." if len(msg.get('content', '')) > 120 else f"\n[{role}]: {content}")

# Verify termination
last = history[-1].get("content", "") if history else ""
assert "FINAL_ANSWER:" in last, "Chat did not terminate cleanly — check max_round and system messages"
print("\nTermination condition met ✓")

## Step 6 — Exercises

### Exercise 1: speaker_selection_method
Change `speaker_selection_method` to `"round_robin"`. Re-run the chat.
Does the conversation still reach `FINAL_ANSWER:`? What happens to quality and round count?

### Exercise 2: Add a fact-checker agent
Create a fifth `AssistantAgent` named `Fact_Checker` that verifies claims the Literature_Reviewer makes.
Add it to the `agents` list between reviewer and critic.

### Exercise 3: Custom termination
Add a second termination condition: stop if the Critic mentions `"no significant gaps"`.
Combine with the existing `FINAL_ANSWER:` check using `or`.

### Exercise 4: Token budget
Reduce `max_tokens` to `256`. How does this affect the quality of each agent's output?
What breaks first?

In [ ]:
# Scratch cell — use for exercises


## Step 7 — Reflection

1. What is `is_termination_msg` and who evaluates it?
2. Why does `human_input_mode="NEVER"` matter for automated pipelines?
3. What's the difference between `speaker_selection_method="auto"` and `"round_robin"`?
4. What happens if no agent ever says `FINAL_ANSWER:` and `max_round` is reached?

*(Double-click to edit)*

1. 
2. 
3. 
4. 